# 06 · Road-damage DETECTOR (YOLO) — the project's trained image model

> **REAL PUBLIC DATA workflow** (read in place from `/kaggle/input`; nothing is downloaded to a laptop or committed; the bundle holds model weights and aggregate metrics only).

Trains a detector for **D40 pothole** and the crack classes **D00 / D10 / D20** on real road photos, evaluates it on **held-out groups** (whole index blocks / source videos — never random frames), checks it on Maharashtra phone photos it never saw, and exports **ONNX** for the backend (`AI_VISION_ONNX_PATH`).

**How to run**
1. *Accelerator* **GPU T4 ×2** (a single GPU or P100 also works), *Internet* **On** (clones the repo, installs `ultralytics`, downloads the pretrained weights and the small AMP-check model). With two GPUs the notebook trains with Ultralytics DDP (`device=[0, 1]`); `CIVIC_NUM_GPUS=1` forces one GPU. The **total** batch is fixed per model size and split evenly (32 = 2 × 16 for `yolov8s`). Checklist and troubleshooting: `docs/KAGGLE_RUNBOOK.md`.
2. *Add Input*: the RDD2020 copy you made, `aliabdelmenam/rdd-2022`, `surbhisaswatimohanty/bharatpothole`, and your Mumbai/Nashik copy — the same four as notebook 05.
3. **First run with `SMOKE = True`** (settings cell): ~10–15 minutes, 1 epoch on a small subset, proves every cell works (data assembly, training, evaluation, ONNX export, parity check). Paste me its output.
4. Then set `SMOKE = False` and use **Save & Run All (Commit)** to train overnight (`TRAIN_HOURS` caps the wall-clock; the notebook finishes evaluation + export afterwards). Download the bundle from the Output tab.

**What this is / is not.** Real images, human-annotated boxes, group-safe splits, per-country and per-source metrics, a domain check on photos of a different kind (Maharashtra). It is NOT a claim about real-world accuracy of the full CivicConnect pipeline, and the licences travel with the model (see the cell *Provenance and licences*): RDD2020 is **non-commercial**, the RDD2022 copy is applied as **CC BY-SA 4.0**, BharatPotHole is **UNVERIFIED** (set `USE_BHARATPOTHOLE = False` to keep it out), and Ultralytics' code is **AGPL-3.0**.

In [ ]:
# --- 0. Locate the CivicConnect repo and set up a work directory (works on Kaggle and locally) ---------------
import importlib.util
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

def find_repo() -> Path:
    """Return a writable directory that contains ai/inference/schemas.py."""
    marker = Path("ai/inference/schemas.py")
    if os.environ.get("CIVIC_REPO"):                                   # explicit override
        return Path(os.environ["CIVIC_REPO"])
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:                                    # running from inside the repo
        if (p / marker).exists():
            return p
    if ON_KAGGLE:                                                      # attached Kaggle Dataset containing the repo or just ai/
        for hit in sorted(Path("/kaggle/input").glob("**/ai/inference/schemas.py"))[:1]:
            src = hit.parents[2]                                       # .../ai/inference/schemas.py -> parent of ai/
            dst = Path("/kaggle/working/repo")
            if not dst.exists():
                shutil.copytree(src, dst)                              # /kaggle/input is read-only
            return dst
        url = os.environ.get("CIVIC_GIT_URL", "https://github.com/tanujb03/CivicConnect_v2")                          # default: clone the project repo's `tanuj` branch (needs Internet ON); override with CIVIC_GIT_URL / CIVIC_GIT_REF
        if url:
            subprocess.run(["git", "clone", "--depth", "1", "--branch", os.environ.get("CIVIC_GIT_REF", "tanuj"), url, "/kaggle/working/repo"], check=True)
            return Path("/kaggle/working/repo")
    raise SystemExit("Cannot find the repo. On Kaggle: turn Internet ON (the notebook clones the project from GitHub) or add a Dataset containing the repo's `ai/` folder (see ai/training/README.md).")

REPO = find_repo()
sys.path.insert(0, str(REPO))
WORK = Path(os.environ.get("CIVIC_WORKDIR") or ("/kaggle/working" if ON_KAGGLE else tempfile.mkdtemp(prefix="civic_work_")))
WORK.mkdir(parents=True, exist_ok=True)

_MODULE = {"scikit-learn": "sklearn", "ultralytics": "ultralytics", "onnxruntime": "onnxruntime", "Pillow": "PIL"}

def ensure(*pkgs):
    """pip install only what is missing (Kaggle images already ship numpy/scipy/scikit-learn/pydantic/httpx)."""
    missing = [p for p in pkgs if importlib.util.find_spec(_MODULE.get(p, p)) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

ensure("numpy", "scipy", "scikit-learn", "pydantic", "httpx")
print(f"repo : {REPO}\nwork : {WORK}\npython {sys.version.split()[0]} | kaggle={ON_KAGGLE}")

## 1 · Settings

In [ ]:
import json

SMOKE = True            # True: ~10-15 min proof run (small subset, 1 epoch, small images). Set False for the overnight run.
MODEL = os.environ.get("CIVIC_YOLO_MODEL", "yolov8s.pt")    # pretrained detector to fine-tune (downloaded on first use); "yolov8n.pt" = faster/smaller, "yolov8m.pt" = slower/stronger
IMGSZ = 640
from ai.training.src import hardware as hw  # noqa: E402

BATCH = hw.yolo_total_batch(MODEL)      # TOTAL batch over all GPUs, fixed per model size (CIVIC_YOLO_BATCH overrides): n 64, s 32, m 16, l 16, x 8. AutoBatch (-1) is not allowed with several GPUs.
EPOCHS = 80             # upper bound; TRAIN_HOURS stops earlier
TRAIN_HOURS = 8.0       # wall-clock cap for training (Kaggle sessions end at 12 h; evaluation + export need ~30 min)
SEED = 42
USE_RDD2020, USE_RDD2022, USE_BHARATPOTHOLE = True, True, True
COUNTRIES = None        # e.g. ["India"] to train on India only; None = every country in the RDD copies
VAL_FRACTION = TEST_FRACTION = 0.1          # group-safe: whole blocks / videos go to val or test
PATHS = json.loads(os.environ.get("CIVIC_PATHS_JSON", "{}"))      # only if discovery by name fails, e.g. {"rdd2020": "/kaggle/input/my-rdd2020"}
if SMOKE:
    IMGSZ, BATCH, EPOCHS, TRAIN_HOURS = 320, 16, 1, None
    MODEL = os.environ.get("CIVIC_YOLO_MODEL", "yolov8n.pt")
print({k: v for k, v in dict(SMOKE=SMOKE, MODEL=MODEL, IMGSZ=IMGSZ, BATCH=BATCH, EPOCHS=EPOCHS, TRAIN_HOURS=TRAIN_HOURS, COUNTRIES=COUNTRIES).items()})

## 2 · Find the inputs, install, check the GPU

In [ ]:
from ai.training.src.data_sources import kaggle_run as kr

CFG = kr.RunConfig(work_dir=WORK, paths=PATHS)
FOUND = {}
for row in kr.plan(CFG, ("rdd2020", "rdd2022", "bharatpothole", "mumbai_nashik_road_surface")):
    print(("FOUND   " if row["found"] else "MISSING ") + f"{row['dataset_id']:28s} {row['path'] or ''}")
    if row["found"]:
        FOUND[row["dataset_id"]] = row["path"]
need = [d for d, on in (("rdd2020", USE_RDD2020), ("rdd2022", USE_RDD2022), ("bharatpothole", USE_BHARATPOTHOLE)) if on and d not in FOUND]
assert not need, f"attach these datasets (or set PATHS / USE_*): {need}"

# GPUs are counted with nvidia-smi / CUDA_VISIBLE_DEVICES: this process must NOT initialise CUDA before Ultralytics launches its DDP workers.
GPUS = hw.plan_gpus()                                                    # CIVIC_NUM_GPUS=1 forces a single GPU, =0 the CPU
DEVICE = os.environ.get("CIVIC_YOLO_DEVICE") or hw.ultralytics_device(GPUS)      # [0, 1] | 0 | "cpu"   (CIVIC_YOLO_DEVICE is a CI hook only)
N_GPU = hw.device_gpu_count(DEVICE)
EVAL_DEVICE = hw.single_device(DEVICE)                                   # evaluation and ONNX export run once, in this process, on one device
BP = hw.split_batch(BATCH, N_GPU, must_divide=True)                      # Ultralytics does batch // n_gpus: make it divide, never silently shrink
BATCH, WORKERS = BP.effective, hw.yolo_workers(N_GPU)
ensure("ultralytics", "onnxruntime", "Pillow", *(["cloudpickle"] if N_GPU > 1 else []))
print(hw.banner(GPUS, BP, WORKERS), f"| device={DEVICE}")
if not SMOKE:
    assert N_GPU, "set Accelerator = GPU for the real run"

## 3 · Provenance and licences — these go into the model card

In [ ]:
from ai.training.src.data_sources.card_schema import load_card

PROV = {}
for ds, on in (("rdd2020", USE_RDD2020), ("rdd2022", USE_RDD2022), ("bharatpothole", USE_BHARATPOTHOLE)):
    if not on:
        continue
    c = load_card(ds)
    PROV[ds] = {"licence": c.license.name, "licence_status": c.license.status, "commercial_use": c.license.commercial_use, "citation": c.citation, "attribution": c.license.attribution}
    print(f"{ds:14s} {c.license.status:17s} {c.license.name}\n               commercial use: {c.license.commercial_use}")
print("\nUltralytics (training/export code and the YOLO architecture): AGPL-3.0. Fine for the hackathon; a commercial deployment needs a different licence/model family.")

## 4 · Class ids of the RDD2022 copy — decoded from evidence, never guessed
The Kaggle YOLO copy of RDD2022 ships **numeric** class ids and no class list. Its India images are (also) in the RDD2020 VOC copy, whose files carry the class **names**. Boxes of the images present in both are matched and each id is named by agreement; if agreement is not near-perfect the RDD2022 copy is **not used**.

In [ ]:
from pathlib import Path

from ai.training.src.vision import yolo_data as yd

ITEMS, STATS = [], {}
ID_MAP_2022 = None
if USE_RDD2020:
    a, STATS["rdd2020"] = yd.collect_rdd_voc(FOUND["rdd2020"], countries=COUNTRIES or ["India"], source="rdd2020")      # the VOC copy used here is the India subset
    ITEMS += a
    print("rdd2020 (VOC, India):", STATS["rdd2020"])
if USE_RDD2022:
    if not (USE_RDD2020 and STATS["rdd2020"]["images"]):
        raise SystemExit("decoding the RDD2022 ids needs the RDD2020 VOC copy: set USE_RDD2020 = True (or USE_RDD2022 = False)")
    dec = yd.decode_from_overlap(ITEMS, FOUND["rdd2022"], countries=["India"])
    print("overlapping images:", dec["overlapping_images"], "| matched boxes:", dec["matched_boxes"])
    for cid, r in dec["per_id"].items():
        print(f"  id {cid}: {r['matched_boxes']:6d} boxes -> {r['names']}  (purity {r['purity']})")
    if dec["ok"]:
        ID_MAP_2022 = {int(k): v for k, v in dec["mapping"].items()}
        print("DECODED:", ID_MAP_2022)
    else:
        print("NOT decodable with confidence -> RDD2022 is left out of training (nothing is assumed).")

## 5 · Assemble one group-safe YOLO dataset (read-only inputs; images are symlinks)

In [ ]:
import json

if USE_RDD2022 and ID_MAP_2022:
    b, STATS["rdd2022"] = yd.collect_yolo(FOUND["rdd2022"], ID_MAP_2022, source="rdd2022", group_fn=yd.rdd_group, countries=COUNTRIES)
    ITEMS += b
    print("rdd2022 (YOLO):", STATS["rdd2022"])
if USE_BHARATPOTHOLE:
    c, STATS["bharatpothole"] = yd.collect_yolo(FOUND["bharatpothole"], {0: "D40"}, source="bharatpothole", group_fn=yd.bharat_group)   # its data.yaml lists ONE class: pothole
    ITEMS += c
    print("bharatpothole (YOLO):", STATS["bharatpothole"])
assert ITEMS, "no training items collected"
if SMOKE:
    step = max(1, len(ITEMS) // 900)
    ITEMS = ITEMS[::step]
DATA = WORK / "road_damage_ds"
MANIFEST = yd.assemble(DATA, ITEMS, seed=SEED, val=VAL_FRACTION, test=TEST_FRACTION)
print(json.dumps(MANIFEST, indent=1)[:2500])
assert MANIFEST["groups_straddling_splits"] == 0 and MANIFEST["split_counts"].get("test", 0) > 0

## 6 · Train

In [ ]:
from ultralytics import YOLO  # noqa: I001

from ai.training.src.vision import ddp

RUNS = WORK / "runs"
model = YOLO(MODEL)
args = dict(data=str(DATA / "data.yaml"), imgsz=IMGSZ, batch=BATCH, epochs=EPOCHS, seed=SEED, project=str(RUNS), name="road_damage", exist_ok=True,
            workers=WORKERS, patience=15, plots=False, amp=True, cache=False, device=DEVICE)
if TRAIN_HOURS:
    args["time"] = TRAIN_HOURS          # wall-clock cap (overrides epochs)
if os.environ.get("CIVIC_YOLO_DEVICE"):          # CI hook only (CPU run on invented data); on Kaggle this stays unset
    args.update(workers=0, amp=False)
if MODEL.endswith(".yaml"):                       # CI hook only: architecture from scratch (no weight download)
    args["pretrained"] = False
with ddp.session(N_GPU, WORK / "gpu_watch.txt") as run:      # prints per-GPU utilisation 2 and 15 min in (no second cell can run meanwhile) and forwards the DDP workers' log into this cell
    model.train(**args)      # several GPUs: Ultralytics launches torch.distributed.run workers; this process waits, then reloads best.pt (validation of the val split ran on rank 0)
TRAIN_SECONDS = run.seconds
BEST = RUNS / "road_damage" / "weights" / "best.pt"
print("best weights:", BEST, BEST.exists())

## 7 · Evaluate on the held-out TEST groups — overall, per class, per country and per source

In [ ]:
from ai.training.src.vision import report as rp

best = YOLO(str(BEST))
DEV = dict(device=EVAL_DEVICE)                  # one device, from this (main) process, after training
ev = rp.evaluate_test_groups(best, DATA, WORK, imgsz=IMGSZ, batch=BATCH, dev=DEV)
print(json.dumps(ev, indent=1))

## 8 · Domain check on photos the model never saw — Maharashtra (Mumbai/Nashik)
Not training data, no boxes: only folder names. For each scene folder, the share of sampled images where the detector reports a **pothole (D40) at confidence ≥ 0.25**. Folders named *Pothole …* should be high; others show the false-alarm side. RAW / Final / Rotated hold the same photos, so only `Final` is sampled.

In [ ]:
MN = rp.domain_check(best, FOUND["mumbai_nashik_road_surface"], imgsz=IMGSZ, dev=DEV, seed=SEED) if "mumbai_nashik_road_surface" in FOUND else {}
for scene, v in MN.items():
    print(f"{scene[:44]:46s} n={v['sampled']:3d}  D40 detected in {v['pothole_detected_share']:.0%}   (folder name says pothole: {v['folder_says_pothole']})")
if not MN:
    print("Mumbai/Nashik not attached (or no 'Final' folders): domain check skipped")

## 9 · Export ONNX, check it against the backend's own decoder, write the model card + bundle

In [ ]:
import ultralytics

CPU = dict(device="cpu")        # export + ONNX parity once, here, on the CPU: the graph is device-independent, the backend runs it on CPU, and a GPU device would make Ultralytics pip-install onnxruntime-gpu
onnx_path = Path(best.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=False, **CPU))
OUT = WORK / "road_damage_bundle"
OUT.mkdir(exist_ok=True)
shutil.copy(onnx_path, OUT / "best.onnx")
PARITY = rp.decoder_parity(rp.load_backend_vision(REPO), OUT / "best.onnx", YOLO(str(OUT / "best.onnx"), task="detect"), sorted((DATA / "images" / "test").iterdir()), imgsz=IMGSZ, dev=CPU)
print("backend decoder vs Ultralytics on the exported ONNX:", PARITY)

EPOCH_SECONDS = rp.epoch_seconds(RUNS / "road_damage" / "results.csv")
HARDWARE = hw.TrainingHardware(gpu_count=N_GPU, world_size=max(N_GPU, 1), gpu_names=list(GPUS.names[:N_GPU]), parallel="DDP" if N_GPU > 1 else "none", batch_effective=BATCH,
                               batch_per_gpu=BP.per_gpu, workers_per_gpu=WORKERS, wall_seconds=TRAIN_SECONDS, epoch_seconds=EPOCH_SECONDS,
                               note="Ultralytics DDP; held-out test evaluation ran once on one GPU and the ONNX export once on the CPU, both in the notebook process" if N_GPU > 1 else "").as_dict()

CARD = {"model_name": "civicconnect-road-damage-yolo", "version": ("smoke-" if SMOKE else "") + rp.sha256(OUT / "best.onnx")[:10], "architecture": MODEL,
        "framework": f"ultralytics {ultralytics.__version__} (AGPL-3.0)", "imgsz": IMGSZ, "classes": yd.CLASSES, "class_meaning": yd.CLASS_MEANING,
        "civic_mapping": {k: list(v) for k, v in yd.CIVIC_MAPPING.items()}, "smoke_run": SMOKE,
        "training": {"epochs_cap": EPOCHS, "hours_cap": TRAIN_HOURS, "seed": SEED, "batch": BATCH, "countries": COUNTRIES, "hardware": HARDWARE},
        "data": {"manifest": MANIFEST, "collection_stats": STATS, "rdd2022_id_decoding": (dec if USE_RDD2022 else None)}, "provenance_and_licences": PROV,
        "evaluation_on_heldout_test_groups": ev, "maharashtra_domain_check": MN, "backend_decoder_parity": PARITY,
        "limits": ["Detects pothole and three crack types in road-survey style photos; says nothing about other civic issues.",
                   "Trained on dashcam/drone/smartphone road-survey images: citizen close-ups, night shots and wet roads are under-represented.",
                   "A detection is a PROPOSAL the citizen confirms; it never creates or closes a case.",
                   "Licences travel with the weights: RDD2020 is non-commercial; RDD2022 applied as CC BY-SA 4.0; BharatPotHole licence unverified; Ultralytics is AGPL-3.0.",
                   "Smoke runs are pipeline proofs only." if SMOKE else "Metrics are on held-out groups of the same sources, not a promise about unseen cities."]}
BUNDLE = rp.write_bundle(WORK, OUT, CARD, [BEST, RUNS / "road_damage" / "results.csv"])
print("bundle:", BUNDLE, f"({BUNDLE.stat().st_size / 1e6:.1f} MB)\nsha256:", rp.sha256(BUNDLE))
print("\nHand-off: unzip into ai/artifacts/road_damage/ and set AI_VISION_ONNX_PATH=ai/artifacts/road_damage/road_damage/best.onnx (backend needs onnxruntime + Pillow).")